In [ ]:
import os
import scanpy as sc
import anndata as ad
import pandas as pd

base_path = "/mnt/..........data/loom/"
samples = ['MoCK1', 'MoCK2', 'MoCK3', 'Zikv1', 'Zikv2', 'Zikv3']
adatas = []

for s in samples:
    file_path = os.path.join(base_path, f"{s}.loom")
    print(f"Reading and processing file: {file_path}")
    
    try:
        adata_temp = sc.read_loom(file_path)
        
        # Enforce unique gene identifiers
        adata_temp.var_names_make_unique()
        
        print(f"Sample {s} layers: {list(adata_temp.layers.keys())}")
        
        # Normalize scVelo specific layer naming conventions
        if "spliced" not in adata_temp.layers and "Spliced" in adata_temp.layers:
            adata_temp.layers["spliced"] = adata_temp.layers["Spliced"]
        if "unspliced" not in adata_temp.layers and "Unspliced" in adata_temp.layers:
            adata_temp.layers["unspliced"] = adata_temp.layers["Unspliced"]
            
        # Synchronize main data matrix with the standardized spliced expression layer
        if "spliced" in adata_temp.layers:
            adata_temp.X = adata_temp.layers["spliced"].copy()
        
        # Homogenize barcode formatting strings
        adata_temp.obs_names = [f"{s}_{i.split(':')[-1].strip('x')}" for i in adata_temp.obs_names]
        
        # Metadata allocation
        adata_temp.obs['Sample'] = s
        adata_temp.obs['Group'] = 'MOCK' if 'MoCK' in s else 'ZIKV'
        
        adatas.append(adata_temp)
        
    except Exception as e:
        print(f"Error parsing sample {s}: {e}")

# Verify cohort integrity before execution of inner join concatenation
if len(adatas) == len(samples):
    print("All samples successfully verified. Initializing inner join concatenation...")
    adata_velo = ad.concat(adatas, join="inner")
    adata_velo.obs_names_make_unique()
    
    # Optional mapping for legacy layer protection if required by manual matrices
    if 'matrix' in adata_velo.layers:
        adata_velo.layers['counts'] = adata_velo.layers['matrix'].copy()
        
    print(f"Concatenation complete. Dimensions: Cells={adata_velo.n_obs} | Common Genes={adata_velo.n_vars}")
    
    save_path = "/mnt/d/WY_single_cell_analysis/data/proceed/scVelo/adata_loom_combined.h5ad"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    adata_velo.write(save_path)
    print(f"Synchronized output synchronized to disk: {save_path}")
else:
    print(f"Concatenation aborted. Pipeline received only {len(adatas)} out of {len(samples)} valid datasets.")